# CNN Regression: CC vs NOCC Comparison

This notebook trains the same CNN model on the zucchini dataset twice:
1. **CC** (Color Corrected) images
2. **NOCC** (No Color Correction) images

Then compares the performance between both to assess the impact of color correction.

## 1. Import Libraries

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from glob import glob
import copy
from PIL import Image
from tqdm import tqdm
from skimage import io
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
mpl.rcParams['figure.figsize'] = 16, 8
plt.rcParams['font.size'] = 10

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Dataset paths
BASE_PATH = r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT'

CC_PATH = os.path.join(BASE_PATH, "cc", "CSVS01")
NOCC_PATH = os.path.join(BASE_PATH, "nocc", "CSVS01")

# Score classes (Score=01 to Score=08)
SCORE_FOLDERS = [f"Score={i:02d}" for i in range(1, 9)]

# Image sizes to compare
IMG_SIZES = [64, 128, 256]

# Training parameters
BATCH_SIZE = 32
NUM_EPOCHS = 1000
LEARNING_RATE = 0.001
TRAIN_SPLIT = 0.8
EARLY_STOPPING_PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.5
DEPTH = 16

# Cross-validation
K_FOLDS = 5

# Composite score weights (used for best model selection)
# composite = w_r2 * R² + w_spearman * Spearman_ρ + w_mae * (1 - MAE/7)
COMPOSITE_WEIGHTS = {'r2': 0.4, 'spearman': 0.4, 'mae': 0.2}

print(f"CC path:   {CC_PATH}")
print(f"NOCC path: {NOCC_PATH}")
print(f"Score folders: {SCORE_FOLDERS}")
print(f"Image sizes to test: {IMG_SIZES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"LR: {LEARNING_RATE}")
print(f"LR scheduler: patience={LR_PATIENCE}, factor={LR_FACTOR}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Cross-validation: {K_FOLDS}-Fold Stratified")
print(f"Composite score weights: {COMPOSITE_WEIGHTS}")

## 3. Data Loading

In [ ]:
def load_dataset(root_path, score_folders):
    """Load image paths and targets from Score=XX folders."""
    data_list = []
    for folder_name in score_folders:
        # Extract score number from folder name (Score=01 -> 1)
        score = int(folder_name.split('=')[1])
        folder_path = os.path.join(root_path, folder_name)
        
        if not os.path.exists(folder_path):
            print(f"  Warning: {folder_path} not found")
            continue
        
        image_paths = glob(os.path.join(folder_path, "*.tif"))
        for img_path in image_paths:
            data_list.append((img_path, float(score)))
        
        print(f"  {folder_name}: {len(image_paths)} images")
    
    return data_list

print("Loading CC dataset:")
cc_data = load_dataset(CC_PATH, SCORE_FOLDERS)
print(f"  Total: {len(cc_data)} images\n")

print("Loading NOCC dataset:")
nocc_data = load_dataset(NOCC_PATH, SCORE_FOLDERS)
print(f"  Total: {len(nocc_data)} images")

## 4. Dataset & Model Definition

In [ ]:
class ImageRegressionDataset(Dataset):
    def __init__(self, data_list, img_size, transform=None, target_noise_std=0.0):
        self.data_list = data_list
        self.img_size = img_size
        self.transform = transform
        self.target_noise_std = target_noise_std
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        img_path, target = self.data_list[idx]
        try:
            img_array = io.imread(img_path)
            if img_array.dtype != np.uint16:
                raise ValueError(f"Image not 16-bit: {img_array.dtype}")
            img_array = img_array.astype(np.float32) / 65535.0
            image = torch.from_numpy(img_array.transpose(2, 0, 1)).float()
            if self.transform:
                image = self.transform(image)
            target = torch.tensor([target], dtype=torch.float32)
            if self.target_noise_std > 0:
                target = target + torch.randn_like(target) * self.target_noise_std
            return image, target
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            image = torch.zeros(3, self.img_size, self.img_size)
            return image, torch.tensor([target], dtype=torch.float32)


class RegressionCNN(nn.Module):
    def __init__(self, input_channels=3, img_height=64, img_width=64, depth=16):
        super(RegressionCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            # Block 1: depth
            nn.Conv2d(input_channels, depth, 3, padding=1), nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, 3, padding=1), nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2: depth*2
            nn.Conv2d(depth, depth*2, 3, padding=1), nn.BatchNorm2d(depth*2), nn.ReLU(),
            nn.Conv2d(depth*2, depth*2, 3, padding=1), nn.BatchNorm2d(depth*2), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3: depth*4
            nn.Conv2d(depth*2, depth*4, 3, padding=1), nn.BatchNorm2d(depth*4), nn.ReLU(),
            nn.Conv2d(depth*4, depth*4, 3, padding=1), nn.BatchNorm2d(depth*4), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 4: depth*8
            nn.Conv2d(depth*4, depth*8, 3, padding=1), nn.BatchNorm2d(depth*8), nn.ReLU(),
            nn.Conv2d(depth*8, depth*8, 3, padding=1), nn.BatchNorm2d(depth*8), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 5: depth*16
            nn.Conv2d(depth*8, depth*16, 3, padding=1), nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.Conv2d(depth*16, depth*16, 3, padding=1), nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 6: depth*32
            nn.Conv2d(depth*16, depth*32, 3, padding=1), nn.BatchNorm2d(depth*32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.flattened_size = depth * img_height * img_width // 128
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size // 2),
            nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4),
            nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1),
            nn.ELU()
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


def get_transforms(img_size):
    """Create train and val transforms with enhanced data augmentation."""
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        # Geometric augmentation
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomChoice([
            transforms.Lambda(lambda x: torch.rot90(x, k=0, dims=[1, 2])),
            transforms.Lambda(lambda x: torch.rot90(x, k=1, dims=[1, 2])),
            transforms.Lambda(lambda x: torch.rot90(x, k=2, dims=[1, 2])),
            transforms.Lambda(lambda x: torch.rot90(x, k=3, dims=[1, 2]))
        ]),
        transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        # Normalize
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    return train_transform, val_transform


def compute_composite_score(r2, spearman, mae, weights=COMPOSITE_WEIGHTS):
    """
    Composite score combining R², Spearman ρ, and normalized MAE.
    Higher is better. MAE normalized by max possible error (7 for scores 1-8).
    """
    mae_normalized = 1.0 - (mae / 7.0)
    return weights['r2'] * r2 + weights['spearman'] * spearman + weights['mae'] * mae_normalized


print("Dataset class, model, transforms, and composite score defined.")
print(f"  Train augmentation: Flip, Rot90, Affine(±15°, ±5%, 0.9-1.1x)")
print(f"  Composite score: {COMPOSITE_WEIGHTS['r2']}×R² + {COMPOSITE_WEIGHTS['spearman']}×ρ + "
      f"{COMPOSITE_WEIGHTS['mae']}×(1-MAE/7)")

## 5. Training Pipeline Function

In [ ]:
def train_single_fold(train_list, val_list, img_size, fold_id="", seed=SEED):
    """
    Train one fold. Uses composite score for best model tracking.
    Returns dict with: model, history, val_preds, val_targets, metrics, composite_score
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    train_transform, val_transform = get_transforms(img_size)
    
    # Create datasets and loaders
    train_dataset = ImageRegressionDataset(train_list, img_size, transform=train_transform, target_noise_std=0.15)
    val_dataset = ImageRegressionDataset(val_list, img_size, transform=val_transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Create model
    model = RegressionCNN(input_channels=3, img_height=img_size, img_width=img_size, depth=DEPTH).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)
    
    # Training loop
    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_r2': [], 'val_spearman': [], 'val_composite': [], 'lr': []}
    best_composite = -float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(NUM_EPOCHS):
        # Train
        model.train()
        train_loss_sum = 0.0
        for images, targets_batch in train_loader:
            images, targets_batch = images.to(device), targets_batch.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets_batch)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * images.size(0)
        train_loss = train_loss_sum / len(train_dataset)
        
        # Validate
        model.eval()
        val_loss_sum = 0.0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for images, targets_batch in val_loader:
                images, targets_batch = images.to(device), targets_batch.to(device)
                outputs = model(images)
                loss = criterion(outputs, targets_batch)
                val_loss_sum += loss.item() * images.size(0)
                all_preds.extend(outputs.cpu().numpy().flatten())
                all_targets.extend(targets_batch.cpu().numpy().flatten())
        
        val_loss = val_loss_sum / len(val_dataset)
        val_mae = mean_absolute_error(all_targets, all_preds)
        val_r2 = r2_score(all_targets, all_preds)
        val_spearman = spearmanr(all_targets, all_preds).correlation
        val_composite = compute_composite_score(val_r2, val_spearman, val_mae)
        current_lr = optimizer.param_groups[0]['lr']
        
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        history['val_r2'].append(val_r2)
        history['val_spearman'].append(val_spearman)
        history['val_composite'].append(val_composite)
        history['lr'].append(current_lr)
        
        # Best model tracking (COMPOSITE SCORE — higher is better)
        if val_composite > best_composite:
            best_composite = val_composite
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        # Print every 20 epochs
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"    {fold_id}Ep {epoch+1:4d} | Loss: {val_loss:.4f} | MAE: {val_mae:.3f} | "
                  f"R²: {val_r2:.3f} | ρ: {val_spearman:.3f} | CS: {val_composite:.4f} | LR: {current_lr:.1e}")
        
        # Early stopping
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"    {fold_id}Early stop at epoch {epoch+1} (best composite: {best_composite:.4f})")
            break
    
    # Load best model and final evaluation
    model.load_state_dict(best_model_state)
    model.eval()
    final_preds, final_targets = [], []
    with torch.no_grad():
        for images, targets_batch in val_loader:
            images = images.to(device)
            outputs = model(images)
            final_preds.extend(outputs.cpu().numpy().flatten())
            final_targets.extend(targets_batch.numpy().flatten())
    
    final_preds = np.array(final_preds)
    final_targets = np.array(final_targets)
    
    metrics = {
        'MSE': mean_squared_error(final_targets, final_preds),
        'RMSE': np.sqrt(mean_squared_error(final_targets, final_preds)),
        'MAE': mean_absolute_error(final_targets, final_preds),
        'R2': r2_score(final_targets, final_preds),
        'Spearman': spearmanr(final_targets, final_preds).correlation,
        'Composite': compute_composite_score(
            r2_score(final_targets, final_preds),
            spearmanr(final_targets, final_preds).correlation,
            mean_absolute_error(final_targets, final_preds)
        ),
        'Epochs': len(history['train_loss']),
    }
    
    return {
        'model': model,
        'history': history,
        'val_preds': final_preds,
        'val_targets': final_targets,
        'metrics': metrics,
    }


def train_model_cv(data_list, img_size, model_name="model", seed=SEED):
    """
    Stratified K-Fold cross-validation training pipeline.
    Returns: dict with fold_results, mean/std metrics, best fold model, aggregated predictions
    """
    print(f"  {K_FOLDS}-Fold Stratified Cross-Validation | {img_size}x{img_size}")
    
    # Prepare arrays for stratified split
    paths = [p for p, _ in data_list]
    targets = np.array([t for _, t in data_list])
    target_bins = targets.astype(int)  # Score classes for stratification
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
    
    fold_results = []
    all_val_preds = []
    all_val_targets = []
    all_val_indices = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(paths, target_bins)):
        print(f"\n  --- Fold {fold_idx+1}/{K_FOLDS} (train={len(train_idx)}, val={len(val_idx)}) ---")
        
        train_list = [data_list[i] for i in train_idx]
        val_list = [data_list[i] for i in val_idx]
        
        fold_seed = seed + fold_idx
        result = train_single_fold(
            train_list, val_list, img_size, 
            fold_id=f"F{fold_idx+1} ", seed=fold_seed
        )
        fold_results.append(result)
        all_val_preds.extend(result['val_preds'].tolist())
        all_val_targets.extend(result['val_targets'].tolist())
        all_val_indices.extend(val_idx.tolist())
        
        m = result['metrics']
        print(f"    Fold {fold_idx+1} → R²={m['R2']:.4f} | ρ={m['Spearman']:.4f} | "
              f"MAE={m['MAE']:.4f} | Composite={m['Composite']:.4f}")
        
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Aggregate metrics across folds
    metric_keys = ['MSE', 'RMSE', 'MAE', 'R2', 'Spearman', 'Composite', 'Epochs']
    mean_metrics = {k: np.mean([f['metrics'][k] for f in fold_results]) for k in metric_keys}
    std_metrics = {k: np.std([f['metrics'][k] for f in fold_results]) for k in metric_keys}
    
    # Select best fold by composite score
    best_fold_idx = np.argmax([f['metrics']['Composite'] for f in fold_results])
    best_model = fold_results[best_fold_idx]['model']
    best_history = fold_results[best_fold_idx]['history']
    
    # Overall metrics on aggregated OOF predictions
    oof_preds = np.array(all_val_preds)
    oof_targets = np.array(all_val_targets)
    oof_metrics = {
        'MSE': mean_squared_error(oof_targets, oof_preds),
        'RMSE': np.sqrt(mean_squared_error(oof_targets, oof_preds)),
        'MAE': mean_absolute_error(oof_targets, oof_preds),
        'R2': r2_score(oof_targets, oof_preds),
        'Spearman': spearmanr(oof_targets, oof_preds).correlation,
        'Composite': compute_composite_score(
            r2_score(oof_targets, oof_preds),
            spearmanr(oof_targets, oof_preds).correlation,
            mean_absolute_error(oof_targets, oof_preds)
        ),
    }
    
    print(f"\n  ═══ {model_name} CV Summary ═══")
    print(f"    Mean R²:       {mean_metrics['R2']:.4f} ± {std_metrics['R2']:.4f}")
    print(f"    Mean Spearman: {mean_metrics['Spearman']:.4f} ± {std_metrics['Spearman']:.4f}")
    print(f"    Mean MAE:      {mean_metrics['MAE']:.4f} ± {std_metrics['MAE']:.4f}")
    print(f"    Mean Composite:{mean_metrics['Composite']:.4f} ± {std_metrics['Composite']:.4f}")
    print(f"    OOF R²:        {oof_metrics['R2']:.4f} | OOF ρ: {oof_metrics['Spearman']:.4f}")
    print(f"    Best fold: {best_fold_idx+1} (Composite={fold_results[best_fold_idx]['metrics']['Composite']:.4f})")
    
    return {
        'fold_results': fold_results,
        'mean_metrics': mean_metrics,
        'std_metrics': std_metrics,
        'oof_metrics': oof_metrics,
        'oof_preds': oof_preds,
        'oof_targets': oof_targets,
        'best_model': best_model,
        'best_history': best_history,
        'best_fold_idx': best_fold_idx,
    }

print("Training functions defined (single fold + K-Fold CV with composite score).")

## 6. Train All Configurations (CC & NOCC × 64, 128, 256) with K-Fold CV

In [ ]:
# Store all results: all_results[img_size] = {'CC': {...}, 'NOCC': {...}}
all_results = {}

for img_size in IMG_SIZES:
    print("\n" + "="*80)
    print(f"  IMAGE SIZE: {img_size}x{img_size}")
    print("="*80)
    
    all_results[img_size] = {}
    
    # Train CC with K-Fold CV
    print(f"\n{'─'*40}")
    print(f"  CC (Color Corrected) | {img_size}x{img_size}")
    print(f"{'─'*40}")
    all_results[img_size]['CC'] = train_model_cv(cc_data, img_size=img_size, model_name=f"CC_{img_size}")
    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Train NOCC with K-Fold CV
    print(f"\n{'─'*40}")
    print(f"  NOCC (No Color Correction) | {img_size}x{img_size}")
    print(f"{'─'*40}")
    all_results[img_size]['NOCC'] = train_model_cv(nocc_data, img_size=img_size, model_name=f"NOCC_{img_size}")
    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n" + "="*80)
print("ALL TRAINING COMPLETE!")
print("="*80)

## 7. Cross-Validation Results Summary

In [ ]:
# Build comprehensive results table with mean ± std from CV
rows = []
for img_size in IMG_SIZES:
    for mode in ['CC', 'NOCC']:
        res = all_results[img_size][mode]
        mm = res['mean_metrics']
        sm = res['std_metrics']
        oof = res['oof_metrics']
        rows.append({
            'Image Size': f"{img_size}x{img_size}",
            'Mode': mode,
            'MAE (mean±std)': f"{mm['MAE']:.4f}±{sm['MAE']:.4f}",
            'RMSE (mean±std)': f"{mm['RMSE']:.4f}±{sm['RMSE']:.4f}",
            'R² (mean±std)': f"{mm['R2']:.4f}±{sm['R2']:.4f}",
            'ρ (mean±std)': f"{mm['Spearman']:.4f}±{sm['Spearman']:.4f}",
            'Composite (mean±std)': f"{mm['Composite']:.4f}±{sm['Composite']:.4f}",
            'OOF R²': oof['R2'],
            'OOF ρ': oof['Spearman'],
            'OOF Composite': oof['Composite'],
            'Mean Epochs': mm['Epochs'],
        })

results_df = pd.DataFrame(rows)
print("="*80)
print(f"FULL RESULTS: CC vs NOCC — {K_FOLDS}-Fold Stratified Cross-Validation")
print("="*80)
print(results_df[['Image Size', 'Mode', 'R² (mean±std)', 'ρ (mean±std)', 
                   'MAE (mean±std)', 'Composite (mean±std)', 'OOF R²']].to_string(index=False))
print("="*80)

# Per-size winner (by OOF composite score)
print("\nPer-size winner (by OOF Composite Score):")
for img_size in IMG_SIZES:
    cc_cs = all_results[img_size]['CC']['oof_metrics']['Composite']
    nocc_cs = all_results[img_size]['NOCC']['oof_metrics']['Composite']
    winner = "CC" if cc_cs > nocc_cs else "NOCC"
    print(f"  {img_size}x{img_size}: {winner} (CC={cc_cs:.4f}, NOCC={nocc_cs:.4f})")

## 8. Training Curves (Best Fold)

In [ ]:
fig, axes = plt.subplots(len(IMG_SIZES), 3, figsize=(18, 5*len(IMG_SIZES)))

for row, img_size in enumerate(IMG_SIZES):
    cc_h = all_results[img_size]['CC']['best_history']
    nocc_h = all_results[img_size]['NOCC']['best_history']
    cc_ep = range(1, len(cc_h['train_loss'])+1)
    nocc_ep = range(1, len(nocc_h['train_loss'])+1)
    
    # Val Loss
    axes[row, 0].plot(cc_ep, cc_h['val_loss'], 'b-', lw=2, label='CC (best fold)')
    axes[row, 0].plot(nocc_ep, nocc_h['val_loss'], 'r-', lw=2, label='NOCC (best fold)')
    axes[row, 0].set_title(f'{img_size}x{img_size} — Val Loss', fontweight='bold')
    axes[row, 0].set_xlabel('Epoch'); axes[row, 0].set_ylabel('Loss')
    axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)
    
    # Val R²
    axes[row, 1].plot(cc_ep, cc_h['val_r2'], 'b-', lw=2, label='CC (best fold)')
    axes[row, 1].plot(nocc_ep, nocc_h['val_r2'], 'r-', lw=2, label='NOCC (best fold)')
    axes[row, 1].set_title(f'{img_size}x{img_size} — Val R²', fontweight='bold')
    axes[row, 1].set_xlabel('Epoch'); axes[row, 1].set_ylabel('R²')
    axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)
    
    # Val Composite Score
    axes[row, 2].plot(cc_ep, cc_h['val_composite'], 'b-', lw=2, label='CC (best fold)')
    axes[row, 2].plot(nocc_ep, nocc_h['val_composite'], 'r-', lw=2, label='NOCC (best fold)')
    axes[row, 2].set_title(f'{img_size}x{img_size} — Val Composite Score', fontweight='bold')
    axes[row, 2].set_xlabel('Epoch'); axes[row, 2].set_ylabel('Composite')
    axes[row, 2].legend(); axes[row, 2].grid(True, alpha=0.3)

plt.suptitle(f'Training Curves (Best Fold): CC vs NOCC — {K_FOLDS}-Fold CV', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Out-of-Fold Predictions vs True — All Sizes

In [ ]:
fig, axes = plt.subplots(len(IMG_SIZES), 2, figsize=(14, 6*len(IMG_SIZES)))

for row, img_size in enumerate(IMG_SIZES):
    for col, (mode, color) in enumerate([('CC', 'blue'), ('NOCC', 'red')]):
        res = all_results[img_size][mode]
        preds = res['oof_preds']
        targets = res['oof_targets']
        oof = res['oof_metrics']
        
        axes[row, col].scatter(targets, preds, alpha=0.6, s=60, c=color, edgecolors='black', linewidth=0.5)
        axes[row, col].plot([1, 8], [1, 8], 'k--', lw=2, label='Perfect')
        axes[row, col].set_xlabel('True Score', fontweight='bold')
        axes[row, col].set_ylabel('Predicted Score', fontweight='bold')
        axes[row, col].set_title(f'{mode} {img_size}x{img_size} (OOF)\n'
                                  f'R²={oof["R2"]:.4f} | ρ={oof["Spearman"]:.4f} | CS={oof["Composite"]:.4f}',
                                  fontweight='bold')
        axes[row, col].legend(); axes[row, col].grid(True, alpha=0.3)
        axes[row, col].set_xlim([0, 9]); axes[row, col].set_ylim([0, 9])

plt.suptitle(f'Out-of-Fold Predictions — {K_FOLDS}-Fold CV', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. OOF Metrics Bar Charts — Size Comparison

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
metrics_list = [('MAE', 'lower'), ('RMSE', 'lower'), ('R2', 'higher'), ('Spearman', 'higher'), ('Composite', 'higher')]
x = np.arange(len(IMG_SIZES))
width = 0.35

for ax, (metric, direction) in zip(axes, metrics_list):
    cc_vals = [all_results[s]['CC']['oof_metrics'][metric] for s in IMG_SIZES]
    nocc_vals = [all_results[s]['NOCC']['oof_metrics'][metric] for s in IMG_SIZES]
    
    bars1 = ax.bar(x - width/2, cc_vals, width, label='CC', color='steelblue', edgecolor='black')
    bars2 = ax.bar(x + width/2, nocc_vals, width, label='NOCC', color='salmon', edgecolor='black')
    
    ax.set_xlabel('Image Size', fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} ({direction} is better)', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{s}x{s}' for s in IMG_SIZES])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.suptitle(f'OOF Metrics: CC vs NOCC — {K_FOLDS}-Fold CV', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save All Results

In [ ]:
# Save all results
output_dir = os.path.join(BASE_PATH, 'trainings')
os.makedirs(output_dir, exist_ok=True)

# Save comprehensive CV results CSV
results_df.to_csv(os.path.join(output_dir, 'cc_vs_nocc_cv_results.csv'), index=False)

# Save per-config: OOF predictions, best fold history, best model
for img_size in IMG_SIZES:
    for mode in ['CC', 'NOCC']:
        prefix = f"{mode.lower()}_{img_size}"
        res = all_results[img_size][mode]
        
        # OOF predictions (all samples, out-of-fold)
        pd.DataFrame({
            'true': res['oof_targets'], 
            'predicted': res['oof_preds']
        }).to_csv(os.path.join(output_dir, f'{prefix}_oof_predictions.csv'), index=False)
        
        # Best fold training history
        pd.DataFrame(res['best_history']).to_csv(
            os.path.join(output_dir, f'{prefix}_best_fold_history.csv'), index=False)
        
        # Per-fold metrics
        fold_metrics_rows = []
        for i, fr in enumerate(res['fold_results']):
            row = {'fold': i+1}
            row.update(fr['metrics'])
            fold_metrics_rows.append(row)
        pd.DataFrame(fold_metrics_rows).to_csv(
            os.path.join(output_dir, f'{prefix}_fold_metrics.csv'), index=False)
        
        # Best model weights
        torch.save(res['best_model'].state_dict(), 
                   os.path.join(output_dir, f'{prefix}_best_model.pth'))

print(f"All results saved to: {output_dir}")
print(f"  - cc_vs_nocc_cv_results.csv (summary with mean±std)")
print(f"  - [cc|nocc]_[64|128|256]_oof_predictions.csv")
print(f"  - [cc|nocc]_[64|128|256]_best_fold_history.csv")
print(f"  - [cc|nocc]_[64|128|256]_fold_metrics.csv")
print(f"  - [cc|nocc]_[64|128|256]_best_model.pth")

In [ ]:
# Per-class summary statistics using OOF predictions
print("=" * 80)
print("PER-CLASS SUMMARY STATISTICS (Out-of-Fold Predictions)")
print("=" * 80)

for img_size in IMG_SIZES:
    for mode in ['CC', 'NOCC']:
        res = all_results[img_size][mode]
        preds = res['oof_preds']
        targets = res['oof_targets']

        results_df_exp = pd.DataFrame({
            'true_value': targets,
            'predicted_value': preds,
            'absolute_error': np.abs(preds - targets),
            'squared_error': (preds - targets) ** 2
        })

        print(f"\n{'='*60}")
        print(f"  {mode} @ {img_size}x{img_size} ({K_FOLDS}-fold OOF)")
        print(f"{'='*60}")

        for target_cls in sorted(results_df_exp['true_value'].unique()):
            subset = results_df_exp[results_df_exp['true_value'] == target_cls]
            mean_pred = subset['predicted_value'].mean()
            std_pred = subset['predicted_value'].std()
            mean_ae = subset['absolute_error'].mean()
            print(f"  Class {int(target_cls):2d}: n={len(subset):3d} | "
                  f"mean_pred={mean_pred:.3f} +/- {std_pred:.3f} | MAE={mean_ae:.3f}")

        om = res['oof_metrics']
        mse = np.mean(results_df_exp['squared_error'])
        rmse = np.sqrt(mse)
        print(f"\n  OVERALL: MSE={mse:.4f}  RMSE={rmse:.4f}  "
              f"MAE={om['MAE']:.4f}  R2={om['R2']:.4f}  "
              f"rho={om['Spearman']:.4f}  CS={om['Composite']:.4f}")

        # Save CSV
        out_dir = os.path.join(BASE_PATH, 'trainings')
        os.makedirs(out_dir, exist_ok=True)
        results_df_exp.to_csv(
            os.path.join(out_dir, f'{mode.lower()}_{img_size}_oof_class_results.csv'),
            index=False)

print("\n" + "=" * 80)
print("Per-class results saved.")

In [ ]:
# Predict/collect all images for frieze visualization
print("Loading images for frieze visualization...")

frieze_data = {}
for img_size in IMG_SIZES:
    frieze_data[img_size] = {}
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    for mode, data in [('CC', cc_data), ('NOCC', nocc_data)]:
        imgs = []
        for img_path, _ in data:
            img_array = io.imread(img_path).astype(np.float32) / 65535.0
            t = torch.from_numpy(img_array.transpose(2, 0, 1)).float()
            t = val_tf(t)
            imgs.append(t.permute(1, 2, 0).numpy())  # Convert to (H, W, C)
        frieze_data[img_size][mode] = np.array(imgs)
        print(f"  {mode} @ {img_size}x{img_size}: {len(imgs)} images loaded")

print("All images loaded for frieze visualization.")

## 12. Draw frieze of predictions

In [ ]:
# Frieze of images arranged according to OOF predictions (floor rounding)
for img_size in IMG_SIZES:
    for mode in ['CC', 'NOCC']:
        all_images = frieze_data[img_size][mode]
        all_preds = all_results[img_size][mode]['oof_preds']

        sorted_indices = np.argsort(all_preds)
        sorted_images = all_images[sorted_indices]
        sorted_preds = all_preds[sorted_indices]

        n = len(sorted_images)
        fig, ax = plt.subplots(1, n + 9, figsize=(n + 9, 1))
        class_int = 0
        for i, (img, pred) in enumerate(zip(sorted_images, sorted_preds)):
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            class_int_tmp = int(np.floor(pred))
            if class_int_tmp > class_int:
                ax[i+class_int].axis('off')
                ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30,
                                     fontweight='bold', ha='center', va='center')
                while class_int < class_int_tmp:
                    class_int += 1
                    ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                    ax[i+class_int].imshow(img)
                    ax[i+class_int].axis('off')
            else:
                ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                ax[i+class_int].imshow(img)
                ax[i+class_int].axis('off')
        while class_int < 9:
            class_int += 1
            if n + class_int - 1 < len(ax):
                ax[n+class_int-1].axis('off')
                ax[n+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30,
                                       fontweight='bold', ha='center', va='center')
        plt.suptitle(f'Frieze by Predictions (floor) -- {mode} @ {img_size}x{img_size}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
# Frieze of images arranged according to OOF predictions (round rounding)
for img_size in IMG_SIZES:
    for mode in ['CC', 'NOCC']:
        all_images = frieze_data[img_size][mode]
        all_preds = all_results[img_size][mode]['oof_preds']

        sorted_indices = np.argsort(all_preds)
        sorted_images = all_images[sorted_indices]
        sorted_preds = all_preds[sorted_indices]

        n = len(sorted_images)
        fig, ax = plt.subplots(1, n + 9, figsize=(n + 9, 1))
        class_int = 0
        for i, (img, pred) in enumerate(zip(sorted_images, sorted_preds)):
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            class_int_tmp = int(np.round(pred))
            if class_int_tmp > class_int:
                ax[i+class_int].axis('off')
                ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30,
                                     fontweight='bold', ha='center', va='center')
                while class_int < class_int_tmp:
                    class_int += 1
                    ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                    ax[i+class_int].imshow(img)
                    ax[i+class_int].axis('off')
            else:
                ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                ax[i+class_int].imshow(img)
                ax[i+class_int].axis('off')
        while class_int < 9:
            class_int += 1
            if n + class_int - 1 < len(ax):
                ax[n+class_int-1].axis('off')
                ax[n+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30,
                                       fontweight='bold', ha='center', va='center')
        plt.suptitle(f'Frieze by Predictions (round) -- {mode} @ {img_size}x{img_size}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 13. Draw frieze of annotations

In [ ]:
# Frieze of images arranged according to true annotations
for img_size in IMG_SIZES:
    for mode in ['CC', 'NOCC']:
        all_images = frieze_data[img_size][mode]
        all_true = all_results[img_size][mode]['oof_targets']

        sorted_indices = np.argsort(all_true)
        sorted_images = all_images[sorted_indices]
        sorted_true = all_true[sorted_indices]

        n = len(sorted_images)
        fig, ax = plt.subplots(1, n + 9, figsize=(n + 9, 1))
        class_int = 0
        for i, (img, true_val) in enumerate(zip(sorted_images, sorted_true)):
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            class_int_tmp = int(np.floor(true_val))
            if class_int_tmp > class_int:
                ax[i+class_int].axis('off')
                ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30,
                                     fontweight='bold', ha='center', va='center')
                while class_int < class_int_tmp:
                    class_int += 1
                    ax[i+class_int].set_title(f'{true_val:.2f}', fontsize=12)
                    ax[i+class_int].imshow(img)
                    ax[i+class_int].axis('off')
            else:
                ax[i+class_int].set_title(f'{true_val:.2f}', fontsize=12)
                ax[i+class_int].imshow(img)
                ax[i+class_int].axis('off')
        while class_int < 9:
            class_int += 1
            if n + class_int - 1 < len(ax):
                ax[n+class_int-1].axis('off')
                ax[n+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30,
                                       fontweight='bold', ha='center', va='center')
        plt.suptitle(f'Frieze by Annotations -- {mode} @ {img_size}x{img_size}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()